In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2_e2e_model/checkpoint-525", device_map='auto')
tokenizer.pad_token = tokenizer.eos_token

model.eval()

max_length = 128
encoded = tokenizer(["name : Blue Spice | Type : pub | food : Chinese | area : riverside | family friendly : no | near : Rainbow Vegetarian Café" + tokenizer.bos_token + " In the riverside near the Rainbow Vegetarian Café is a family friendly Chinese pub named the Blue Spice ." + tokenizer.eos_token, "name : The Cricketers | Type : restaurant | customer rating : average | family friendly : yes | near : Café Sicilia" + tokenizer.bos_token + "Near Café Sicilia , there is an average - rated , children friendly restaurant called The Cricketers ." + tokenizer.eos_token], truncation=True, padding="max_length", max_length=max_length, return_tensors='pt').to('cuda')

c:\Users\3un8i\anaconda3\envs\IDA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
example_input_ids = encoded['input_ids']
batch_size = example_input_ids.shape[0]
example_attention_mask = encoded['attention_mask']
example_position_ids = torch.arange(0, max_length, dtype=torch.long, device='cuda').unsqueeze(0).expand(batch_size, max_length) # 각 시퀀스마다 0에서 시작해서 1씩 증가하는 배열

labels = example_input_ids.clone().to('cuda')
for b in range(batch_size):
    for i in range(max_length):
        if labels[b][i] == 50256:
            labels[b][i] = -100
            break
        else:
            labels[b][i] = -100
    for i in range(max_length - 1, -1, -1):
        if labels[b][i - 1] != 50256:
            break
        else:
            labels[b][i] = -100

valid_indices = labels != -100

loss = torch.tensor(0.0, dtype=torch.float32, device='cuda')

for i in range(1, max_length): # auto-regressive decoding loop
    input_ids = example_input_ids[:, 0:i]
    attention_mask = example_attention_mask[:, 0:i]
    position_ids = example_position_ids[:, 0:i]
    model_inputs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'position_ids': position_ids, 'use_cache': False} # 'use_cache': False 설정으로 KV caching은 끄고 진행함

    inputs_embeds = model.transformer.wte(input_ids) # Word Token Embeddings: vocab_sz=50257 가지의 토큰을 d_model=768 차원 벡터로 변환
    position_embeds = model.transformer.wpe(position_ids)
    hidden_states = inputs_embeds + position_embeds
    hidden_states = model.transformer.drop(hidden_states)

    for l in range(len(model.transformer.h)): # block 12개에 대한 반복문
        block = model.transformer.h[l]

        # Self-Attention 시작
        residual = hidden_states
        hidden_states = block.ln_1(hidden_states) # layer normalization
        
        query, key, value = block.attn.c_attn(hidden_states).split(768, dim=2) 
        # GPT2는 c_attn이라는 행렬에다 W_Q, W_K, W_V를 합쳐놨음

        query = block.attn._split_heads(query, 12, 64) 
        key = block.attn._split_heads(key, 12, 64)
        value = block.attn._split_heads(value, 12, 64)
        # d_model=768은 12개의 head로 나뉨. 각각 768/12=64차원

        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query,
            key,
            value,
            attn_mask=None,
            dropout_p=block.attn_dropout.p if block.training else 0.0,
            is_causal=True,
        ) # 어텐션 스코어 행렬 계산

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, i, 768)
        # 12개 head의 출력들 다시 합치기 

        attn_output = block.attn.c_proj(attn_output) # 여기가 W_O
        attn_output = block.attn.resid_dropout(attn_output)

        hidden_states = attn_output + residual
        # Self-Attention 끝
        # Feed-Forward Network 시작
        residual = hidden_states
        hidden_states = block.ln_2(hidden_states) # layer normalization

        feed_forward_hidden_states = block.mlp.c_fc(hidden_states) # W_fc1: 768차원에서 3072차원으로
        feed_forward_hidden_states = block.mlp.act(feed_forward_hidden_states) # 활성화함수 GELU
        feed_forward_hidden_states = block.mlp.c_proj(feed_forward_hidden_states) # W_fc2: 3072차원에서 768차원으로 
        feed_forward_hidden_states = block.mlp.dropout(feed_forward_hidden_states)

        hidden_states = residual + feed_forward_hidden_states
        # Feed-Forward Network 끝

    transformer_outputs = model.transformer.ln_f(hidden_states)
    transformer_outputs = transformer_outputs.view((-1, i, 768))

    outputs = model.lm_head(transformer_outputs)

    next_token_logits = outputs[:, -1, :].clone().float().to('cuda')

    loop_loss = torch.nn.functional.cross_entropy(next_token_logits, labels[:, i], ignore_index=-100, reduction='sum')
    if valid_indices[:, i].sum().item() > 0:
        loss += loop_loss

loss /= valid_indices.sum().item()

loss.backward()